Azure provides several storage services beyond Blob Storage, each designed for a specific workload:

- **Azure Files** — fully managed SMB and NFS file shares in the cloud, mountable on Windows, Linux, and macOS
- **Azure Managed Disks** — block storage volumes attached to VMs (OS disks and data disks)
- **Azure Queue Storage** — simple message queuing between application components
- **Azure Table Storage** — serverless NoSQL key-value store for structured, non-relational data

All four services live inside a **Storage Account**, giving them the same redundancy, encryption, and network security options as Blob Storage.

## Azure Files

**Azure Files** is a fully managed file share service accessible over **SMB 3.x** (Windows/Linux/macOS) and **NFS 4.1** (Linux/macOS). It replaces on-premises file servers — lift-and-shift legacy applications, provide shared configuration or home directories, or use it as a persistent volume in AKS.

### Share tiers

| Tier | Backed by | IOPS | Latency | Use case |
|---|---|---|---|---|
| **Transaction Optimised** | HDD | Baseline | ~10 ms | General-purpose; high-transaction workloads |
| **Hot** | HDD | Baseline | ~10 ms | Collaboration, shared home directories |
| **Cool** | HDD | Baseline | ~10 ms | Online archival; infrequent access |
| **Premium** | SSD | Up to 100K | < 1 ms | Latency-sensitive: databases, dev tools |

Premium file shares require a **Premium File Share** storage account (`FileStorage` kind) — they cannot be created in a Standard GPv2 account.

### Protocols

| Protocol | OS support | Port | Notes |
|---|---|---|---|
| **SMB 3.x** | Windows, Linux, macOS | 445 | Encryption in transit; default for Windows |
| **NFS 4.1** | Linux, macOS | 2049 | Premium shares only; requires VNet integration (no public NFS) |
| **REST API** | Any | 443 | HTTPS; used by Azure File Sync agent |

> **Port 445** is blocked by many ISPs and corporate firewalls. For on-premises access, use **Azure File Sync** or a **VPN/ExpressRoute** connection.

### Azure File Sync

**Azure File Sync** turns an Azure file share into a distributed, cached file system:

```
Azure Files (cloud endpoint)
      ↕  sync
File Sync agent (Windows Server)
      ↕  local SMB
On-premises clients
```

- Install the File Sync agent on one or more Windows Servers
- Files are cached locally on each server — access is fast even over WAN
- **Cloud tiering** frees local disk space by replacing infrequently accessed files with a pointer; Azure downloads them on demand
- Multiple servers in different offices sync the same share — replacing a traditional DFS setup

### Authentication for Azure Files

| Method | Use case |
|---|---|
| **Entra ID (Kerberos)** | Domain-joined VMs; users authenticate with their Entra ID identity |
| **On-prem AD DS (Kerberos)** | Hybrid — on-prem AD-joined machines access Azure Files with existing AD accounts |
| **Storage account key** | Mount from non-domain-joined systems; service accounts; scripted mounts |
| **SAS** | Time-limited programmatic access via REST |

## Azure Managed Disks

**Managed Disks** are block storage volumes for Azure VMs — the Azure equivalent of AWS EBS. "Managed" means Azure handles storage account placement, redundancy, and capacity — you just create a disk and attach it.

### Disk types

| Type | Media | Max IOPS | Max throughput | Use case |
|---|---|---|---|---|
| **Ultra Disk** | SSD | 400,000 | 10,000 MB/s | SAP HANA, top-tier SQL, real-time analytics |
| **Premium SSD v2** | SSD | 80,000 | 1,200 MB/s | Most production databases; configurable IOPS/throughput |
| **Premium SSD** | SSD | 20,000 | 900 MB/s | Production databases, enterprise apps |
| **Standard SSD** | SSD | 6,000 | 750 MB/s | Web servers, lightly loaded apps, dev/test |
| **Standard HDD** | HDD | 500 | 60 MB/s | Backups, infrequent access, cold storage |

### Disk roles

| Role | Notes |
|---|---|
| **OS disk** | Contains the OS — tagged `/dev/sda` on Linux; max 4 TiB |
| **Temporary disk** | Local SSD on the physical host; not replicated; wiped on VM stop/restart |
| **Data disk** | Additional volumes attached to the VM; persists independently of the VM |

> The **temporary disk** (`/mnt` on Linux, `D:` on Windows) is not managed storage — it is directly attached local NVMe/SSD. Never store data that must survive a VM restart here.

### Disk redundancy

| Option | Description |
|---|---|
| **LRS** | Default — 3 copies in one data centre |
| **ZRS** | 3 copies across 3 zones — zone-resilient; available for Premium SSD and Standard SSD |

Ultra Disk and Premium SSD v2 support **zone pinning** — the disk is placed in a specific AZ alongside the VM.

### Disk snapshots and images

- **Snapshot** — point-in-time copy of a managed disk; used for backup or creating new disks
- **Incremental snapshot** — only the changes since the last snapshot; much cheaper for large disks
- **Image** — capture a generalised (sysprepped) VM into a reusable template stored in **Azure Compute Gallery**

### Disk encryption

| Layer | Description |
|---|---|
| **Azure Storage encryption (SSE)** | Default — 256-bit AES at rest, managed by Azure or CMK via Key Vault |
| **Azure Disk Encryption (ADE)** | OS-level — BitLocker on Windows, dm-crypt on Linux; keys in Key Vault |
| **Encryption at host** | Encrypts temp disk and OS/data disk cache at the VM host level |
| **Confidential disk encryption** | Encryption keys bound to the VM's vTPM — for confidential VMs |

## Azure Queue Storage

**Azure Queue Storage** is a simple, durable message queue service for decoupling application components — the lightweight alternative to Azure Service Bus.

### Characteristics

| Property | Value |
|---|---|
| Max message size | 64 KB |
| Max queue size | 500 TiB |
| Max TTL | 7 days (default); up to infinite |
| Visibility timeout | Message hidden from other consumers while processing |
| At-least-once delivery | Yes — messages may be delivered more than once |
| Ordering | Best-effort FIFO — not guaranteed |

### When to use Queue Storage vs Service Bus

| Feature | Queue Storage | Service Bus |
|---|---|---|
| Message size | Up to 64 KB | Up to 100 MB |
| Ordering | Best-effort | Guaranteed (sessions) |
| Dead-letter queue | No | Yes |
| Topics/subscriptions | No | Yes |
| Transactions | No | Yes |
| At-least-once delivery | Yes | Yes |
| Price | Very low | Higher |

> Use **Queue Storage** for simple, high-volume decoupling where ordering and exactly-once semantics are not required. Use **Service Bus** for enterprise messaging with ordering, dead-lettering, and topics.

## Azure Table Storage

**Azure Table Storage** is a serverless NoSQL key-value store for structured, non-relational data — the predecessor to Azure Cosmos DB Table API.

### Data model

```
Table
 └── Entity (row)
       ├── PartitionKey  (string) — groups related entities; determines physical storage
       ├── RowKey        (string) — unique within a partition
       ├── Timestamp     (auto)   — last-modified time
       └── Properties    (up to 252 custom columns, any OData-supported type)
```

Queries are fast when filtering on **PartitionKey + RowKey** (point query) or **PartitionKey + RowKey range** (range query). Full-table scans are slow and expensive — design your partition key carefully.

### Table Storage vs Cosmos DB Table API

| Feature | Table Storage | Cosmos DB Table API |
|---|---|---|
| Latency | < 10 ms | < 10 ms (single-digit guaranteed) |
| Global distribution | No | Yes |
| SLA | 99.9% | 99.999% |
| Secondary indexes | No | Yes |
| Throughput provisioning | Auto | Manual (RU/s) or autoscale |
| Price | Very low | Higher |

> Use **Cosmos DB Table API** for new workloads requiring global distribution or SLA guarantees. Use **Table Storage** for simple, cost-sensitive scenarios where its limitations are acceptable.

## Storage Account Network Security

All Azure Storage services share the same network security model:

### Firewall and virtual network rules

By default, storage accounts accept traffic from all networks. Restrict this by:
1. Setting **default action to Deny**
2. Adding explicit **Allow** rules for specific IP ranges, VNet subnets (via service endpoints), or trusted Azure services

### Private endpoints

A **private endpoint** assigns the storage account a private IP address within your VNet. DNS resolves `<account>.blob.core.windows.net` to the private IP — traffic stays on the Azure backbone, never on the public internet.

Each storage service (blob, file, queue, table) gets its own private endpoint sub-resource.

### Trusted Azure services bypass

Some Azure services — Azure Backup, Azure Site Recovery, Azure Monitor, Event Grid — need to access storage without going through firewall rules. Enable **"Allow trusted Azure services"** to whitelist these services without opening the firewall broadly.

### Minimum TLS version

Set the minimum TLS version to **1.2** (the default for new accounts). Requests using TLS 1.0 or 1.1 are rejected.

## Choosing the Right Azure Storage Service

| Scenario | Storage service |
|---|---|
| Store files, images, videos, backups, data lake | **Blob Storage (Block Blob)** |
| Streaming / append-only logs | **Blob Storage (Append Blob)** |
| VM OS disk and data volumes | **Managed Disks** |
| Shared file system (SMB/NFS) across VMs or on-prem servers | **Azure Files** |
| Extend on-prem file server to Azure, cache locally | **Azure File Sync** |
| Simple async message queue between services | **Queue Storage** |
| Enterprise messaging with ordering, dead-letter, topics | **Service Bus** |
| Key-value NoSQL store, simple queries | **Table Storage** |
| Key-value NoSQL with global distribution and SLA | **Cosmos DB Table API** |
| High-performance SSD database volumes on a VM | **Premium SSD / Ultra Disk** |
| Cheap cold storage for infrequently read backups | **Blob Archive tier** |

## Working with Azure Files and Managed Disks via Python

In [ ]:
# pip install azure-identity azure-storage-file-share azure-mgmt-compute azure-mgmt-storage azure-storage-queue

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.storage.fileshare import ShareServiceClient, ShareClient
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.storage import StorageManagementClient

credential       = DefaultAzureCredential()
subscription_id  = "<your-subscription-id>"
resource_group   = "my-rg"
account_name     = "mystorageaccount"
account_url      = f"https://{account_name}.file.core.windows.net"

file_service  = ShareServiceClient(account_url=account_url, credential=credential)
compute       = ComputeManagementClient(credential, subscription_id)
storage_mgmt  = StorageManagementClient(credential, subscription_id)

In [ ]:
# Create a file share (Standard tier)
share = file_service.create_share("my-share", quota=100)  # 100 GiB
print(f"Share created: my-share")

# List shares
for s in file_service.list_shares():
    print(f"  {s.name:<30} {s.quota} GiB  tier: {s.access_tier or 'TransactionOptimised'}")

# Upload a file
share_client = file_service.get_share_client("my-share")
dir_client   = share_client.get_directory_client("configs")
dir_client.create_directory()

file_client = dir_client.get_file_client("app.json")
file_client.upload_file(b'{"env": "production"}')
print("Uploaded configs/app.json")

# Download a file
download = file_client.download_file()
content  = download.readall()
print(f"Downloaded: {content}")

In [ ]:
from azure.mgmt.compute.models import Disk, DiskSku, CreationData

# Create a Premium SSD managed disk
disk_params = Disk(
    location="eastus",
    sku=DiskSku(name="Premium_LRS"),
    creation_data=CreationData(create_option="Empty"),
    disk_size_gb=128
)
poller = compute.disks.begin_create_or_update(resource_group, "my-data-disk", disk_params)
disk   = poller.result()
print(f"Disk created: {disk.name}  size: {disk.disk_size_gb} GiB  tier: {disk.sku.name}")

# List all managed disks in the resource group
for d in compute.disks.list_by_resource_group(resource_group):
    attached_to = d.managed_by.split("/")[-1] if d.managed_by else "unattached"
    print(f"  {d.name:<40} {d.disk_size_gb:>6} GiB  {d.sku.name:<20} {attached_to}")

In [ ]:
from azure.mgmt.compute.models import Snapshot, SnapshotSku, CreationData

# Create an incremental snapshot of a managed disk
source_disk_id = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Compute/disks/my-data-disk"

snap_params = Snapshot(
    location="eastus",
    sku=SnapshotSku(name="Standard_LRS"),
    creation_data=CreationData(
        create_option="Copy",
        source_resource_id=source_disk_id
    ),
    incremental=True   # incremental snapshot — only changed pages
)
poller   = compute.snapshots.begin_create_or_update(resource_group, "my-disk-snap", snap_params)
snapshot = poller.result()
print(f"Snapshot created: {snapshot.name}  incremental: {snapshot.incremental}")

In [ ]:
from azure.storage.queue import QueueServiceClient

queue_url     = f"https://{account_name}.queue.core.windows.net"
queue_service = QueueServiceClient(account_url=queue_url, credential=credential)

# Create a queue and send messages
queue_client = queue_service.create_queue("my-queue")
queue_client = queue_service.get_queue_client("my-queue")
queue_client.send_message("task-001")
queue_client.send_message("task-002")

# Receive and process messages
messages = queue_client.receive_messages(messages_per_page=5)
for msg in messages:
    print(f"Processing: {msg.content}")
    queue_client.delete_message(msg)  # remove after processing

# Check queue length
props = queue_client.get_queue_properties()
print(f"Messages remaining: {props.approximate_message_count}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Azure Files | Managed SMB/NFS file shares — lift-and-shift file servers; Premium tier for low-latency workloads |
| Azure File Sync | Sync Azure file shares to Windows Server; cloud tiering frees local disk |
| Port 445 | Required for SMB — often blocked; use VPN/ExpressRoute or File Sync for on-prem access |
| Managed Disks | Block storage for VMs — OS, temp (ephemeral), and data disks |
| Temporary disk | Local SSD on the VM host — not replicated; wiped on stop/restart; never store persistent data here |
| Ultra Disk | Highest IOPS/throughput — SAP HANA, top-tier databases |
| Premium SSD v2 | Configurable IOPS and throughput — most production databases |
| Incremental snapshot | Only changed pages since last snapshot — much cheaper than full snapshot for large disks |
| Queue Storage | Simple at-least-once message queue; 64 KB max; no ordering guarantee |
| Queue vs Service Bus | Queue for simple high-volume; Service Bus for ordering, dead-letter, topics |
| Table Storage | Serverless NoSQL key-value; fast on PartitionKey+RowKey; no secondary indexes |
| Table vs Cosmos DB | Cosmos DB Table API for global distribution and SLAs; Table Storage for simple low-cost scenarios |
| Private endpoint | Private IP for storage in VNet — per service sub-resource (blob, file, queue, table) |
| Trusted Azure services | Bypass for Azure Backup, Monitor, Event Grid without opening the firewall |